In [1]:
from dataloader_helper import read_file
import numpy as np
import pandas as pd
import vector
import awkward as ak
# plot w2 distance matrix

In [2]:
local_dataset_path = ''

# please create a local_dataset_path.txt file that contains the path to the folder that contains the JetClass Dataset
with open('local_dataset_path.txt') as f:
    local_dataset_path = f.read()

In [3]:
file_names = ['HToBB', 'HToCC', 'HToGG', 'HToWW4Q', 'TTBar', 'WToQQ', 'ZToQQ']

for file in file_names:
    x_particles, _, y = read_file(local_dataset_path + file + '_120.root', 150, particle_features=['part_pt', 'part_eta', 'part_phi', 'part_mass'])
    mini_dataset_x = np.transpose(x_particles[:1000], axes=(0,2,1))
    mini_dataset_y = np.argmax(y[:1000], axis=1)
    
    # since we are skiping over the H->lvqq jet and the default order of the labels is such that H->lvqq is labeled as 5, we subtract 1 for each of the subsequent kinds of jets that comes after 5. 
    mini_dataset_y[mini_dataset_y > 5] -= 1
    
    np.savez(file+'_1k.npz', x=mini_dataset_x, y=mini_dataset_y)

In [4]:
qg_dataset_name = 'train_file_0.parquet'

df = pd.read_parquet('../dataset/'+qg_dataset_name)

In [5]:
px_arr = df['part_px'].values.tolist()
py_arr = df['part_py'].values.tolist()
pz_arr = df['part_pz'].values.tolist()
e_arr = df['part_energy'].values.tolist()

p4 = vector.zip({'px': px_arr, 'py': py_arr, 'pz': pz_arr, 'energy': e_arr})

In [6]:
def _pad(a, maxlen, value=0, dtype='float32'):
    if isinstance(a, np.ndarray) and a.ndim >= 2 and a.shape[1] == maxlen:
        return a
    elif isinstance(a, ak.Array):
        if a.ndim == 1:
            a = ak.unflatten(a, 1)
        a = ak.fill_none(ak.pad_none(a, maxlen, clip=True), value)
        return ak.values_astype(a, dtype)
    else:
        x = (np.ones((len(a), maxlen)) * value).astype(dtype)
        for idx, s in enumerate(a):
            if not len(s):
                continue
            trunc = s[:maxlen].astype(dtype)
            x[idx, :len(trunc)] = trunc
        return x

l = [p4.pt, p4.eta, p4.phi, p4.mass] # check if the results make sense

x_particles = np.stack([ak.to_numpy(_pad(a, maxlen=150)) for a in l], axis=1)

In [7]:
mini_dataset_x = np.transpose(x_particles[:1000], axes=(0,2,1))
mini_dataset_y = np.zeros(1000)

np.savez('QCD_1k.npz', x=mini_dataset_x, y=mini_dataset_y)

In [8]:
# saving the class labels to the npz file
# a dictionary of what the class labels represents to the npz file
class_labels = {
    'QCD'   : 0,
    'HToBB'  : 1, 
    'HToCC'  : 2, 
    'HToGG'  : 3, 
    'HToWW4Q': 4, 
    'TTBar'  : 5, 
    'WToQQ'  : 6, 
    'ZToQQ'  : 7
}

np.savez('class_labels.npz', **class_labels)